In [4]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer

# 1. Đọc dữ liệu và lọc bỏ các dòng thiếu biến mục tiêu (salary_avg)
df = pd.read_csv('../data/05_final_dataset.csv')
df_ml = df.dropna(subset=['salary_avg']).copy()

# 2. XỬ LÝ CỘT LIST (TECH_STACK)
# Hàm chuyển đổi chuỗi text thành list chuẩn
def clean_tech_list(val):
    if pd.isna(val) or val == 'Not Specified' or val == 'Unknown':
        return []
    if isinstance(val, str):
        if val.startswith('['): # Xử lý dạng "['Python', 'Java']"
            try: return [x.strip() for x in ast.literal_eval(val)]
            except: return []
        else: # Xử lý dạng "Python, Java, React"
            return [x.strip() for x in val.split(',')]
    return []

df_ml['tech_stack_clean'] = df_ml['tech_stack'].apply(clean_tech_list)

# Khởi tạo MultiLabelBinarizer để mã hóa list thành các cột 0/1
mlb = MultiLabelBinarizer()
tech_encoded = mlb.fit_transform(df_ml['tech_stack_clean'])

# Tạo DataFrame chứa các cột kỹ năng mới (thêm tiền tố 'tech_' để dễ phân biệt)
tech_df = pd.DataFrame(tech_encoded, columns=[f"tech_{c}" for c in mlb.classes_], index=df_ml.index)

# Ghép các cột kỹ năng vừa tạo trở lại bảng dữ liệu chính
df_ml = pd.concat([df_ml, tech_df], axis=1)

# 3. XỬ LÝ ONE-HOT ENCODING CÁC CỘT PHÂN LOẠI CÒN LẠI
categorical_cols = ['work_method', 'contract_type', 'location', 'education_level', 'job_category']
df_encoded = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True)

# 4. CHUẨN HÓA THANG ĐO (MIN-MAX SCALING) CÁC CỘT SỐ
scaler = MinMaxScaler()
cols_to_scale = ['exp_years', 'job_level'] # salary_avg là biến mục tiêu (Y) thường không cần scale, nhưng tùy Model
df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])

# 5. DỌN DẸP CỘT RÁC VÀ LƯU FILE
# Loại bỏ các cột dạng text không đưa vào mô hình được
cols_to_drop = ['url', 'job_title', 'company_name', 'source', 'salary_min', 'salary_max', 'tech_stack', 'tech_stack_clean']
df_final = df_encoded.drop(columns=cols_to_drop)

df_final.to_csv('../data/final_ml_dataset.csv', index=False)
print("✅ Kích thước tập dữ liệu sẵn sàng đưa vào Model:", df_final.shape)

# Xem thử một vài cột kỹ năng vừa được bóc tách
df_final.head(5)

✅ Kích thước tập dữ liệu sẵn sàng đưa vào Model: (2399, 428)


,language_req,is_manager,exp_years,job_level,is_shift_work,salary_avg,tech_.NET,tech_3ds Max,tech_AWS,tech_AWS CloudWatch,...,job_category_General Developer,job_category_HR/Admin,job_category_IT Sales/Marketing,job_category_IT Support,job_category_Management/Architect,job_category_Mobile Developer,job_category_Other,job_category_PM/PO,job_category_QA/Tester,job_category_System/DevOps/Security
0,1,0,0.083333,0.4,0,13.500000,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,1,0,0.250000,0.4,0,35.000000,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
5,1,0,0.083333,0.4,0,12.500000,0,0,0,0,...,False,False,False,True,False,False,False,False,False,False
6,1,0,0.333333,0.4,0,35.000000,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
8,0,0,0.166667,0.4,0,14.166667,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False


In [5]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer


df_ml = df_final


Q1 = df_ml['salary_avg'].quantile(0.25)
Q3 = df_ml['salary_avg'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_ml = df_ml[(df_ml['salary_avg'] >= lower_bound) & (df_ml['salary_avg'] <= upper_bound)]
print(f"Số lượng dòng sau khi xóa NaN và Outliers: {len(df_final)}")


df_ml.to_csv('../data/final_ml_dataset.csv', index=False)
print("✅ Kích thước tập dữ liệu cuối cùng:", df_ml.shape)

Số lượng dòng sau khi xóa NaN và Outliers: 2399
✅ Kích thước tập dữ liệu cuối cùng: (2307, 428)


In [7]:
# 1. Thay thế dấu chấm (.) thành dấu gạch dưới (_) cho toàn bộ tên cột
df_ml.columns = df_ml.columns.str.replace('.', '_', regex=False)

# (Tùy chọn) Xóa thêm các ký tự đặc biệt khác nếu MongoDB vẫn báo lỗi (ví dụ dấu $)
df_ml.columns = df_ml.columns.str.replace('$', '', regex=False)

# 2. Code mẫu để đẩy dataframe lên MongoDB
import pymongo
import json

# Chuyển DataFrame thành dạng list các dictionary (định dạng chuẩn của MongoDB)
records = df_ml.to_dict(orient='records')

# Kết nối và đẩy lên DB
# client = pymongo.MongoClient("mongodb://localhost:27017/")
# db = client["it_job_market"]
# collection = db["jobs_normalized"]
# collection.insert_many(records)
df_ml.to_csv('../data/final_ml_dataset.csv', index=False)

print("Đã đổi tên cột và sẵn sàng upload!")

Đã đổi tên cột và sẵn sàng upload!
